# Holo Quantum — quantum computing in your browser (Cirq)

The basis for hologram-native quantum experiments: **cirq-core** (Google Cirq's pure-Python core) runs entirely on your hardware via Pyodide/WASM — circuit construction, state-vector & density-matrix simulation, measurement, noise. No server, no cloud backend. Run all cells.

In [ ]:
# Offline setup: cirq-core from the bundled wheel index; its math deps come from the Pyodide dist.
import piplite
await piplite.install(['cirq-core'])
import pyodide_js
await pyodide_js.loadPackage(['sympy', 'networkx', 'matplotlib', 'pandas', 'scipy'])
import cirq, numpy as np
print('cirq', cirq.__version__, '— ready (offline)')

## 1 · A Bell pair — entanglement from H + CNOT

In [ ]:
q = cirq.LineQubit.range(2)
bell = cirq.Circuit([cirq.H(q[0]), cirq.CNOT(q[0], q[1])])
print(bell)
sv = cirq.Simulator().simulate(bell).final_state_vector
print('state :', cirq.dirac_notation(sv))
print('probs :', {f'{i:02b}': round(float(abs(a)**2), 3) for i, a in enumerate(sv)})

## 2 · Measure it — sampling the entangled pair (1000 shots)

In [ ]:
circ = bell + cirq.measure(*q, key='m')
counts = cirq.Simulator().run(circ, repetitions=1000).histogram(key='m')
print({f'{k:02b}': v for k, v in sorted(counts.items())})
print('-> only 00 and 11 appear: the qubits are correlated')

## 3 · A GHZ state — 3-qubit entanglement

In [ ]:
g = cirq.LineQubit.range(3)
ghz = cirq.Circuit([cirq.H(g[0]), cirq.CNOT(g[0], g[1]), cirq.CNOT(g[1], g[2])])
sv = cirq.Simulator().simulate(ghz).final_state_vector
print('GHZ   :', cirq.dirac_notation(sv))

## 4 · A parameterized rotation — ⟨Z⟩ vs θ (should trace cos θ)

In [ ]:
import matplotlib.pyplot as plt
q0 = cirq.LineQubit(0); sim = cirq.Simulator()
thetas = np.linspace(0, 2*np.pi, 60)
expZ = []
for th in thetas:
    sv = sim.simulate(cirq.Circuit([cirq.rx(float(th))(q0)])).final_state_vector
    p0, p1 = abs(sv[0])**2, abs(sv[1])**2
    expZ.append(float(p0 - p1))
plt.figure(figsize=(8,4))
plt.plot(thetas, expZ, 'o', ms=3, label='Cirq sim ⟨Z⟩')
plt.plot(thetas, np.cos(thetas), '-', lw=1, label='cos θ')
plt.xlabel('θ'); plt.ylabel('⟨Z⟩'); plt.legend(); plt.title('rx(θ)|0⟩ expectation'); plt.grid(alpha=.3)
plt.show()

### Basis for hologram-native quantum experiments
- Build any circuit with `cirq` — gates, moments, simulators, noise, density matrices.
- State vectors are plain numpy arrays → content-address them with `holo_gpu.address(...)` for reproducible, O(1)-cached experiments, and offload large state-vector kernels to the **WebGPU** bridge (`holo_gpu.run`) as your systems grow.
- Everything ran on your hardware, served from content-addressed bytes — no server, no cloud QPU.